# Khmer Grammar Checking - Binary Classification Pipeline

This notebook implements the full pipeline for classifying Khmer sentences as grammatically **Right** (1) or **Wrong** (0).

Pipeline steps:
1. POS Tagging using `seanghay/khmer-pos-roberta`
2. Feature extraction (OOV ratio, grammar score, semantic coherence, interaction features)
3. Word embeddings via FastText (`cc.km.300.bin`)
4. BiGRU with Attention Pooling classifier
5. Evaluation & Prediction

## 1. Install Dependencies

In [ ]:
!pip install torch>=2.0.0 transformers>=4.30.0 datasets>=2.0.0 fasttext>=0.9.2 \
    pandas>=1.5.0 numpy>=1.24.0 scikit-learn>=1.2.0 \
    matplotlib>=3.6.0 seaborn>=0.12.0 tqdm>=4.64.0 \
    accelerate>=0.20.0 fastapi>=0.104.0 uvicorn[standard]>=0.24.0

## 2. Imports

In [ ]:
import os
import gc
import ast
import numpy as np
import pandas as pd
from collections import Counter
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import fasttext

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 3. Configuration

In [ ]:
SEED = 42
FASTTEXT_MODEL_PATH = "cc.km.300.bin"
DATA_PATH = "train_data.csv"
MODEL_SAVE_PATH = "gru_model.pth"

EMBEDDING_DIM = 300
HIDDEN_DIM = 256
OUTPUT_DIM = 2
N_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 0.001
N_EPOCHS = 15
BATCH_SIZE = 32
PATIENCE = 3
CLIP_VALUE = 1.0

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

FEATURE_COLUMNS = [
    "oov_ratio",
    "dep_grammar_score",
    "has_complete_clause",
    "semantic_coherence",
    "grammar_oov_interaction",
]

USE_FEATURE_FUSION = True
NUM_EXTRA_FEATURES = len(FEATURE_COLUMNS)

torch.manual_seed(SEED)
np.random.seed(SEED)

## 4. Data Loading

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
print(f"Loaded {len(df)} samples")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:\n{df['sentence_correct'].value_counts()}")

# Parse string representations back into actual lists
if "tokens" in df.columns:
    df["tokens"] = df["tokens"].apply(ast.literal_eval)
if "pos_tags" in df.columns:
    df["pos_tags"] = df["pos_tags"].apply(ast.literal_eval)
print("Parsed tokens and pos_tags from string to list")

print(f"\nFirst 3 rows:")
df[["text", "sentence_correct"]].head(3)

## 5. POS Tagger (Khmer RoBERTa)

In [ ]:
class KhmerPOSTagger:
    def __init__(self, model_name="seanghay/khmer-pos-roberta"):
        print(f"Loading POS tagger: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        self.pipeline = pipeline(
            "token-classification",
            model=self.model,
            tokenizer=self.tokenizer,
            aggregation_strategy="simple",
        )
        print("POS tagger loaded successfully")

    def tag_sentence(self, sentence):
        try:
            tags = self.pipeline(sentence)
            tokens = [t["word"] for t in tags]
            pos_tags = [t["entity_group"] for t in tags]
            return tokens, pos_tags
        except Exception as e:
            print(f"POS tagging error: {e}")
            return [], []

    def tag_dataframe(self, df, text_column="text"):
        print(f"Tagging {len(df)} sentences...")
        tokens_list = []
        pos_tags_list = []
        for sentence in tqdm(df[text_column], desc="POS Tagging"):
            tokens, pos_tags = self.tag_sentence(sentence)
            tokens_list.append(tokens)
            pos_tags_list.append(pos_tags)
        df["tokens"] = tokens_list
        df["pos_tags"] = pos_tags_list
        print("POS tagging complete")
        return df

## 6. Feature Extractors

### 6.1 OOV Calculator (Out-of-Vocabulary ratio using FastText)

In [ ]:
class EmbeddingOOVCalculator:
    def __init__(self, embedding_model=None):
        self.embedding_model = embedding_model
        self.vocab = set()
        if embedding_model:
            self._load_vocab_from_model()

    def _load_vocab_from_model(self):
        try:
            if hasattr(self.embedding_model, "get_words"):
                self.vocab = set(self.embedding_model.get_words())
            else:
                raise AttributeError("Model doesn't have recognized vocabulary interface")
            print(f"Loaded {len(self.vocab):,} words from embedding model")
        except Exception as e:
            print(f"Error loading vocabulary from model: {e}")

    def is_in_vocabulary(self, word):
        return word.strip() in self.vocab

    def calculate_oov_ratio(self, tokens):
        if isinstance(tokens, str):
            words = tokens.split()
        else:
            words = tokens
        if not words:
            return 0.0
        oov_count = sum(1 for word in words if not self.is_in_vocabulary(word))
        return oov_count / len(words)

    def calculate_oov_details(self, tokens):
        if isinstance(tokens, str):
            words = tokens.split()
        else:
            words = tokens
        oov_words = []
        known_words = []
        for word in words:
            if self.is_in_vocabulary(word):
                known_words.append(word)
            else:
                oov_words.append(word)
        return {
            "total_words": len(words),
            "known_words": len(known_words),
            "oov_words": len(oov_words),
            "oov_ratio": len(oov_words) / max(len(words), 1),
            "oov_word_list": oov_words,
            "known_word_list": known_words,
            "vocabulary_coverage": len(known_words) / max(len(words), 1),
        }

    def add_oov_features(self, df, text_col="tokens"):
        df = df.copy()
        oov_ratios = []
        vocab_coverages = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="OOV Analysis"):
            text = row[text_col]
            details = self.calculate_oov_details(text)
            oov_ratios.append(details["oov_ratio"])
            vocab_coverages.append(details["vocabulary_coverage"])
        df["oov_ratio"] = oov_ratios
        df["vocab_coverage"] = vocab_coverages
        print(f"Mean OOV ratio: {df['oov_ratio'].mean():.3f}")
        print(f"Mean vocab coverage: {df['vocab_coverage'].mean():.3f}")
        return df

### 6.2 Grammar Extractor (POS-based)

In [ ]:
class SimplePOSGrammarExtractor:
    @staticmethod
    def calculate_grammar_score(pos_tags):
        if len(pos_tags) < 2:
            return 0.5
        score = 0.5
        has_subject = any(tag.startswith("NN") for tag in pos_tags)
        has_verb = any(tag.startswith("VB") for tag in pos_tags)
        has_object = len([tag for tag in pos_tags if tag.startswith("NN")]) > 1
        if has_subject and has_verb:
            score += 0.3
        if has_object:
            score += 0.2
        return min(score, 1.0)

    @staticmethod
    def has_complete_clause(pos_tags):
        if not pos_tags or len(pos_tags) == 0:
            return 0
        has_noun = any(tag.startswith("NN") or tag.startswith("PR") for tag in pos_tags)
        has_verb = any(tag.startswith("VB") or tag == "AUX" for tag in pos_tags)
        if has_noun and has_verb:
            return 1
        if has_verb and len(pos_tags) >= 1:
            if pos_tags[0].startswith("VB"):
                return 1
            for i in range(len(pos_tags) - 1):
                if pos_tags[i].startswith("VB") and pos_tags[i + 1].startswith("NN"):
                    return 1
        existential_markers = {"VB", "AUX", "CC"}
        has_existential = any(tag in existential_markers for tag in pos_tags)
        if has_existential and has_noun:
            return 1
        if "WP" in pos_tags or "WDT" in pos_tags or "WRB" in pos_tags:
            if has_verb:
                return 1
        if "CC" in pos_tags and has_verb:
            return 1
        if has_verb and len(pos_tags) >= 5:
            return 1
        verb_count = sum(1 for tag in pos_tags if tag.startswith("VB") or tag == "AUX")
        if verb_count >= 2:
            return 1
        has_adj = any(tag.startswith("JJ") for tag in pos_tags)
        if has_noun and has_adj and len(pos_tags) >= 2:
            for i in range(len(pos_tags) - 1):
                if pos_tags[i].startswith("NN") and pos_tags[i + 1].startswith("JJ"):
                    return 1
        return 0

    def extract_features(self, df):
        print("Extracting simple POS-based grammar features...")
        df["dep_grammar_score"] = df["pos_tags"].apply(self.calculate_grammar_score)
        df["has_complete_clause"] = df["pos_tags"].apply(self.has_complete_clause)
        print(f"Mean grammar score: {df['dep_grammar_score'].mean():.3f}")
        return df

### 6.3 Semantic Coherence

In [ ]:
class SemanticCoherence:
    CONTENT_POS = {"NN", "VB", "JJ", "RB", "CD"}

    POS_WEIGHTS = {
        ("NN", "VB"): 1.5,
        ("VB", "NN"): 1.5,
        ("JJ", "NN"): 1.2,
        ("NN", "NN"): 1.0,
        ("VB", "VB"): 1.0,
        ("RB", "VB"): 0.8,
    }

    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.dim = embedding_model.get_dimension()

    def get_vector(self, word):
        try:
            return self.embedding_model.get_word_vector(word)
        except Exception:
            return np.zeros(self.dim)

    def weighted_semantic_coherence(self, tokens, pos_tags):
        if len(tokens) < 2:
            return 0.5
        vectors = np.array([self.get_vector(w) for w in tokens])
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        normalized = vectors / norms
        scores, weights = [], []
        for i in range(len(tokens) - 1):
            w = self.POS_WEIGHTS.get((pos_tags[i], pos_tags[i + 1]), 0.5)
            if w > 0:
                scores.append(np.dot(normalized[i], normalized[i + 1]))
                weights.append(w)
        return np.average(scores, weights=weights) if scores else 0.0

    def content_word_coherence(self, tokens, pos_tags):
        idx = [i for i in range(len(tokens)) if pos_tags[i] in self.CONTENT_POS]
        if len(idx) < 2:
            return 0.5
        vecs = np.array([self.get_vector(tokens[i]) for i in idx])
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        normed = vecs / norms
        sims = [np.dot(normed[i], normed[i + 1]) for i in range(len(normed) - 1)]
        return np.mean(sims)

    def multi_distance_coherence(self, tokens, max_distance=3):
        if len(tokens) < 2:
            return 0.5
        vecs = np.array([self.get_vector(w) for w in tokens])
        norms = np.linalg.norm(vecs, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1, norms)
        normed = vecs / norms
        scores = []
        for d in range(1, min(max_distance + 1, len(tokens))):
            sims = np.sum(normed[:-d] * normed[d:], axis=1) / d
            scores.extend(sims)
        return np.mean(scores) if scores else 0.0

    def ensemble_coherence_score(self, tokens, pos_tags):
        if len(tokens) < 2:
            return 0.5
        return (
            0.4 * self.weighted_semantic_coherence(tokens, pos_tags)
            + 0.3 * self.content_word_coherence(tokens, pos_tags)
            + 0.3 * self.multi_distance_coherence(tokens)
        )

    def extract_features(self, df):
        print("Calculating semantic coherence features...")
        scores = [
            self.ensemble_coherence_score(row["tokens"], row["pos_tags"])
            for _, row in tqdm(df.iterrows(), total=len(df), desc="Semantic Analysis")
        ]
        df["semantic_coherence"] = scores
        print(f"Mean coherence: {df['semantic_coherence'].mean():.3f}")
        return df

### 6.4 Interaction Features

In [ ]:
class InteractionFeatureExtractor:
    @staticmethod
    def calculate_grammar_oov_interaction(grammar_score, oov_ratio):
        return grammar_score * (1 - oov_ratio)

    def extract_features(self, df):
        df["grammar_oov_interaction"] = df.apply(
            lambda r: self.calculate_grammar_oov_interaction(
                r.get("dep_grammar_score", 1.0), r["oov_ratio"]
            ),
            axis=1,
        )
        print("Interaction features extracted")
        return df

### 6.5 Feature Pipeline Orchestrator

In [ ]:
class FeaturePipeline:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.pos_tagger = KhmerPOSTagger()
        self.oov_extractor = EmbeddingOOVCalculator(self.embedding_model)
        self.grammar_extractor = SimplePOSGrammarExtractor()
        self.semantic_extractor = SemanticCoherence(self.embedding_model)
        self.interaction_extractor = InteractionFeatureExtractor()

    def extract_all_features(self, df):
        df = self.pos_tagger.tag_dataframe(df)
        df = self.oov_extractor.add_oov_features(df)
        df = self.grammar_extractor.extract_features(df)
        df = self.semantic_extractor.extract_features(df)
        df["grammar_oov_interaction"] = df.apply(
            lambda r: self.interaction_extractor.calculate_grammar_oov_interaction(
                r.get("dep_grammar_score", 1.0), r["oov_ratio"]
            ),
            axis=1,
        )
        return df

    def create_temp_dataset(self, tokens):
        return KhmerTextDataset([tokens], [0], self.embedding_model, use_cache=True)

## 7. Dataset

In [ ]:
class KhmerTextDataset(Dataset):
    _embedding_cache = {}

    def __init__(self, tokens_list, labels, embedding_model, use_cache=True):
        self.tokens_list = tokens_list
        self.labels = labels
        self.embedding_model = embedding_model
        self.embedding_dim = embedding_model.get_dimension()
        self.use_cache = use_cache
        if self.use_cache:
            self._build_cache()

    def _build_cache(self):
        unique_tokens = set()
        for tokens in self.tokens_list:
            unique_tokens.update(tokens)
        new_tokens = unique_tokens - set(self._embedding_cache.keys())
        if new_tokens:
            print(f"Caching {len(new_tokens)} new unique tokens...")
            for token in new_tokens:
                try:
                    vec = self.embedding_model.get_word_vector(token)
                except Exception:
                    vec = np.zeros(self.embedding_dim)
                self._embedding_cache[token] = vec

    def __len__(self):
        return len(self.tokens_list)

    def __getitem__(self, idx):
        tokens = self.tokens_list[idx]
        label = self.labels[idx]
        embeddings = []
        for token in tokens:
            if self.use_cache and token in self._embedding_cache:
                vec = self._embedding_cache[token]
            else:
                try:
                    vec = self.embedding_model.get_word_vector(token)
                except Exception:
                    vec = np.zeros(self.embedding_dim)
            embeddings.append(vec)
        embeddings = torch.FloatTensor(np.array(embeddings))
        label = torch.LongTensor([label])
        return embeddings, label, len(tokens)

    @classmethod
    def clear_cache(cls):
        cls._embedding_cache.clear()


def collate_batch(batch):
    embeddings_list, labels_list, lengths_list = zip(*batch)
    padded = pad_sequence(embeddings_list, batch_first=True)
    labels = torch.cat(labels_list)
    lengths = torch.LongTensor(lengths_list)
    return padded, labels, lengths

## 8. GRU Model with Attention

In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, dropout=0.5):
        super().__init__()
        self.gru = nn.GRU(
            embedding_dim, hidden_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, embedded_text, text_lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded_text, text_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        hidden = self.dropout(torch.cat((hidden[-2], hidden[-1]), dim=1))
        return self.fc(hidden)


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, sequence_output, lengths):
        scores = self.attention(sequence_output).squeeze(-1)
        batch_size, max_len = sequence_output.size(0), sequence_output.size(1)
        mask = (
            torch.arange(max_len, device=sequence_output.device).unsqueeze(0)
            < lengths.unsqueeze(1)
        )
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), sequence_output).squeeze(1)
        return context, weights


class ImprovedGRUClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers,
                 dropout=0.5, num_extra_features=0, use_feature_fusion=False):
        super().__init__()
        self.use_feature_fusion = use_feature_fusion
        self.num_extra_features = num_extra_features
        self.gru = nn.GRU(
            embedding_dim, hidden_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        self.attention = AttentionPooling(hidden_dim * 2)
        pooled_dim = hidden_dim * 2 * 2
        self.layer_norm = nn.LayerNorm(pooled_dim)
        mlp_input_dim = pooled_dim
        if use_feature_fusion and num_extra_features > 0:
            mlp_input_dim += num_extra_features
        self.classifier = nn.Sequential(
            nn.Linear(mlp_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, embedded_text, text_lengths, extra_features=None):
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded_text, text_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_output, hidden = self.gru(packed)
        sequence_output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        attention_context, _ = self.attention(sequence_output, text_lengths)
        mask = (
            torch.arange(sequence_output.size(1), device=sequence_output.device).unsqueeze(0)
            < text_lengths.unsqueeze(1)
        ).unsqueeze(-1).float()
        mean_output = (sequence_output * mask).sum(dim=1) / text_lengths.unsqueeze(1).float()
        pooled = self.layer_norm(torch.cat([attention_context, mean_output], dim=1))
        pooled = self.dropout(pooled)
        if self.use_feature_fusion and extra_features is not None:
            pooled = torch.cat([pooled, extra_features], dim=1)
        return self.classifier(pooled)

## 9. Training Utilities

In [ ]:
def train_gru_epoch(model, dataloader, optimizer, criterion, device, clip_value=1.0):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    for embeddings, labels, lengths in tqdm(dataloader, desc="Training"):
        embeddings, labels, lengths = embeddings.to(device), labels.to(device), lengths.to(device)
        optimizer.zero_grad()
        predictions = model(embeddings, lengths)
        loss = criterion(predictions, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader), correct / total


def evaluate_gru(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for embeddings, labels, lengths in tqdm(dataloader, desc="Evaluating"):
            embeddings, labels, lengths = embeddings.to(device), labels.to(device), lengths.to(device)
            predictions = model(embeddings, lengths)
            loss = criterion(predictions, labels)
            _, predicted = torch.max(predictions, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            epoch_loss += loss.item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(dataloader), correct / total, all_preds, all_labels


def train_gru_epoch_with_features(model, dataloader, feature_tensor, optimizer, criterion, device, clip_value=1.0):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    batch_start_idx = 0
    for embeddings, labels, lengths in tqdm(dataloader, desc='Training'):
        batch_size = embeddings.size(0)
        embeddings, labels, lengths = embeddings.to(device), labels.to(device), lengths.to(device)
        optimizer.zero_grad()
        extra_features = None
        if feature_tensor is not None:
            extra_features = feature_tensor[batch_start_idx:batch_start_idx + batch_size].to(device)
            batch_start_idx += batch_size
        if hasattr(model, 'use_feature_fusion') and model.use_feature_fusion:
            predictions, _ = model(embeddings, lengths, extra_features)
        else:
            predictions = model(embeddings, lengths)
        loss = criterion(predictions, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_value)
        optimizer.step()
        _, predicted = torch.max(predictions, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        epoch_loss += loss.item()
    return epoch_loss / len(dataloader), correct / total


def evaluate_gru_with_features(model, dataloader, feature_tensor, criterion, device):
    model.eval()
    epoch_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    batch_start_idx = 0
    with torch.no_grad():
        for embeddings, labels, lengths in tqdm(dataloader, desc='Evaluating'):
            batch_size = embeddings.size(0)
            embeddings, labels, lengths = embeddings.to(device), labels.to(device), lengths.to(device)
            extra_features = None
            if feature_tensor is not None:
                extra_features = feature_tensor[batch_start_idx:batch_start_idx + batch_size].to(device)
                batch_start_idx += batch_size
            if hasattr(model, 'use_feature_fusion') and model.use_feature_fusion:
                predictions, _ = model(embeddings, lengths, extra_features)
            else:
                predictions = model(embeddings, lengths)
            loss = criterion(predictions, labels)
            _, predicted = torch.max(predictions, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            epoch_loss += loss.item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return epoch_loss / len(dataloader), correct / total, all_preds, all_labels


class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0, mode="min"):
        assert mode in {"min", "max"}
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        self.num_bad_epochs = 0
        self.early_stop = False
        self.best_state_dict = None
        self.best_epoch = -1
        self._is_improvement = (
            lambda c, b: (b - c) > min_delta if mode == "min" else (c - b) > min_delta
        )

    def step(self, current_value, model=None, epoch=None):
        if self._is_improvement(current_value, self.best):
            self.best = current_value
            self.num_bad_epochs = 0
            self.best_epoch = epoch if epoch is not None else self.best_epoch
            if model is not None:
                self.best_state_dict = {
                    k: v.detach().clone() for k, v in model.state_dict().items()
                }
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.early_stop = True
        return self.early_stop

## 10. Load FastText Embedding Model

In [ ]:
if not os.path.exists(FASTTEXT_MODEL_PATH):
    print(f"FastText model not found at {FASTTEXT_MODEL_PATH}, downloading...")
    import fasttext.util
    fasttext.util.download_model('km', if_exists='ignore')

print("Loading FastText model...")
embedding_model = fasttext.load_model(FASTTEXT_MODEL_PATH)
print(f"Embedding dimension: {embedding_model.get_dimension()}")

## 11. Data Splitting

In [ ]:
available_features = [c for c in FEATURE_COLUMNS if c in df.columns]
X = df[available_features]
y = df["sentence_correct"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Train set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")

## 12. Feature Extraction on Raw Text

If the CSV already has pre-computed features and tokens, this step is optional.
Skip to the next section to use the pre-computed data. Uncomment to run.

In [ ]:
# Uncomment to run feature extraction from scratch on raw text:
# pipeline = FeaturePipeline(embedding_model)
# df_raw = pd.DataFrame({"text": df["text"]})
# df_features = pipeline.extract_all_features(df_raw)
# df_features.head()

## 13. Create Datasets & DataLoaders

In [ ]:
def get_tokens_from_df(df, indices):
    if "tokens" in df.columns:
        return df.loc[indices, "tokens"].tolist()
    return df.loc[indices, "text"].tolist()

train_dataset = KhmerTextDataset(
    get_tokens_from_df(df, X_train.index), y_train.values, embedding_model
)
val_dataset = KhmerTextDataset(
    get_tokens_from_df(df, X_val.index), y_val.values, embedding_model
)
test_dataset = KhmerTextDataset(
    get_tokens_from_df(df, X_test.index), y_test.values, embedding_model
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 14. Train GRU Model

In [ ]:
train_feat_tensor = torch.FloatTensor(X_train_scaled)
val_feat_tensor = torch.FloatTensor(X_val_scaled)

model = ImprovedGRUClassifier(
    EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, DROPOUT,
    num_extra_features=NUM_EXTRA_FEATURES, use_feature_fusion=USE_FEATURE_FUSION,
).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
early_stopper = EarlyStopping(patience=PATIENCE, min_delta=0.0, mode="min")

print("=" * 80)
print("Training Improved GRU Model (with feature fusion)")
print("=" * 80)
print(f"Feature Fusion: {USE_FEATURE_FUSION}, Extra Features: {NUM_EXTRA_FEATURES}")

train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(N_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{N_EPOCHS}" + "-" * 70)
    train_loss, train_acc = train_gru_epoch_with_features(
        model, train_loader, train_feat_tensor, optimizer, criterion, DEVICE, CLIP_VALUE
    )
    val_loss, val_acc, _, _ = evaluate_gru_with_features(
        model, val_loader, val_feat_tensor, criterion, DEVICE
    )
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc * 100:.2f}%")
    early_stopper.step(val_loss, model, epoch=epoch + 1)
    if early_stopper.early_stop:
        print(f"Early stopping at epoch {epoch + 1}, best at epoch {early_stopper.best_epoch}")
        break

if early_stopper.best_state_dict:
    model.load_state_dict(early_stopper.best_state_dict)
    torch.save({
        "model_state_dict": early_stopper.best_state_dict,
        "scaler": scaler,
        "feature_columns": FEATURE_COLUMNS,
    }, MODEL_SAVE_PATH)
    print(f"\nBest model saved to {MODEL_SAVE_PATH}")

print("\n" + "=" * 80)
print("Training Complete!")

### Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, label="Train Loss")
ax1.plot(val_losses, label="Val Loss")
ax1.set_title("Loss Curves")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(train_accs, label="Train Acc")
ax2.plot(val_accs, label="Val Acc")
ax2.set_title("Accuracy Curves")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

## 15. Evaluate on Test Set

In [ ]:
test_feat_tensor = torch.FloatTensor(scaler.transform(X_test))
test_loss, test_acc, y_pred, y_true = evaluate_gru_with_features(model, test_loader, test_feat_tensor, criterion, DEVICE)
test_precision = precision_score(y_true, y_pred)
test_recall = recall_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred)

print(f"\nTest Results:")
print(f"  Accuracy:  {test_acc:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall:    {test_recall:.4f}")
print(f"  F1-Score:  {test_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["Wrong", "Right"]))

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Wrong", "Right"], yticklabels=["Wrong", "Right"])
plt.title("Confusion Matrix - GRU")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

## 16. Prediction Example

In [ ]:
class SentencePredictor:
    def __init__(self, feature_pipeline_instance, trained_model, scaler, feature_columns, model_type="gru"):
        self.feature_pipeline = feature_pipeline_instance
        self.model = trained_model
        self.scaler = scaler
        self.feature_columns = feature_columns
        self.model_type = model_type
        self._cache = {}
        self.device = DEVICE
        self.model.to(self.device)

    def predict_sentence(self, sentence):
        temp_df = pd.DataFrame({"text": [sentence], "sentence_correct": [0]})
        processed_df = self.feature_pipeline.extract_all_features(temp_df)
        features_dict = processed_df[self.feature_columns].iloc[0].to_dict()
        tokens = processed_df["tokens"].iloc[0]
        pos_tags = processed_df["pos_tags"].iloc[0]

        feature_vector = processed_df[self.feature_columns].iloc[0].values.reshape(1, -1)
        feature_vector_scaled = self.scaler.transform(feature_vector)
        extra_features = torch.FloatTensor(feature_vector_scaled).to(self.device)
        temp_dataset = self.feature_pipeline.create_temp_dataset(tokens)

        self.model.eval()
        with torch.no_grad():
            embeddings, _, lengths = temp_dataset[0]
            embeddings = embeddings.unsqueeze(0).to(self.device)
            lengths = torch.LongTensor([lengths]).to(self.device)
            if hasattr(self.model, "use_feature_fusion") and self.model.use_feature_fusion:
                output, _ = self.model(embeddings, lengths, extra_features)
            else:
                output = self.model(embeddings, lengths)
            probs = torch.softmax(output, dim=1)
            confidence, prediction = torch.max(probs, dim=1)
            prediction = prediction.item()
            confidence = confidence.item()

        return {
            "sentence": sentence,
            "prediction": "Right" if prediction == 1 else "Wrong",
            "prediction_numeric": prediction,
            "confidence": confidence,
            "features": features_dict,
            "tokens": tokens,
            "pos_tags": pos_tags,
        }


feature_pipeline = FeaturePipeline(embedding_model)
predictor = SentencePredictor(feature_pipeline, model, scaler, FEATURE_COLUMNS)

test_sentences = [
    "អ៊ីតាលីបានឈ្នះលើព័រទុយហ្គាល់ 31-5 ក្នុងប៉ូលCនៃពីធីប្រកួតពានរង្វាន់ពិភពលោកនៃកីឡាបាល់ឱបឆ្នាំ2007ដែលប្រព្រឹត្តនៅប៉ាសឌេសប្រីនក្រុងប៉ារីសបារាំង។",
    "បាន ស៊ុត នាទីទី ដំបូង ពាក់កណ្តាល សំរាប់ អ៊ីតាលី",
    "ក្មេង ទៅ សាលា",
]

for sent in test_sentences:
    result = predictor.predict_sentence(sent)
    print(f"SEN: {result['sentence'][:50]}...")
    print(f"PRED: {result['prediction']} (conf: {result['confidence']:.4f})")
    print()

## 17. FastAPI Backend (Optional)

To run the API server, save the model first (`gru_model.pth` already saved above), then start:

```bash
uvicorn main:app --reload
```

Or run it inline below.

In [ ]:
# from fastapi import FastAPI
# from pydantic import BaseModel
# import uvicorn

# app = FastAPI(title="Khmer Grammar Checker API")

# class PredictRequest(BaseModel):
#     sentence: str

# @app.post("/predict")
# async def predict(request: PredictRequest):
#     result = predictor.predict_sentence(request.sentence)
#     return result

# @app.get("/health")
# async def health():
#     return {"status": "ok"}

# uvicorn.run(app, host="0.0.0.0", port=8000)

---
**Done!** The full pipeline is complete: feature extraction → GRU training → evaluation → prediction.